In [1]:
import sys
import os
sys.path.append(os.path.abspath(".."))

In [2]:
from part_1_RAG.ingest import load_faq_data

documents = load_faq_data()

In [3]:
documents_llm = []

for doc in documents:
    if doc["course"] == "llm-zoomcamp":
        documents_llm.append(doc)

len(documents_llm)

79

In [4]:
documents = documents_llm

In [5]:
doc = documents[0]
print(doc["id"])
print(doc["question"])
print(doc["answer"])

74eb249bbf
I just discovered the course. Can I still join?
Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.


In [32]:
# not  the best because our model is small
# from pydantic import BaseModel

# class Questions(BaseModel):
#     questions: list[str]

In [ ]:
# data_gen_instructions = """
# You emulate a student who's taking our course.
# Formulate 5 questions this student might ask based on a FAQ record. The record
# should contain the answer to the questions, and the questions should be complete and not too short.
# If possible, use as fewer words as possible from the record.

# The output should resemble how people ask questions
# on the internet. Not too formal, not too short, not too long.
# """.strip()

In [38]:
from openai import OpenAI

ollama_client = OpenAI(
    base_url="http://localhost:11434/v1",
    api_key="ollama"  # required by the SDK but ignored by Ollama
)

In [39]:
import json

user_prompt = json.dumps(doc)

In [ ]:
data_gen_instructions = """
You emulate a student who's taking our course.
Formulate exactly 5 distinct questions this student might ask based on a FAQ record. 

Rules:
1. The record contains the answer to the questions.
2. The questions must be complete sentences and not too short.
3. Use as few words from the record as possible.
4. Outout must resemble casual internet phrasing (not too formal).

Provide all 5 questions in the requested array structure.
""".strip()

In [41]:
messages = [
    {"role": "developer", "content": data_gen_instructions},
    {"role": "user", "content": user_prompt}
]

In [42]:
from pydantic import BaseModel, Field

class Questions(BaseModel):
    # This forces the JSON schema to require exactly 5 elements
    questions: list[str] = Field(
        ..., 
        min_length=5, 
        max_length=5, 
        description="A list containing exactly 5 distinct questions."
    )

In [30]:
response = ollama_client.responses.parse(
    model="qwen3.5:9b",
    input=messages,
    text_format=Questions
)

In [ ]:
# result with Constrain the Schema with Field
result = response.output_parsed

print(result)
print(result.questions)

questions=['I just discovered the course. Can I still join?.', 'I want to join the LLM Zoomcamp now. Is it too late for me to start?', 'Is there a deadline for joining this course?.', 'What happens if I join late but miss the project deadline?', "Does the certificate expire if I don't submit the project immediately?"]
['I just discovered the course. Can I still join?.', 'I want to join the LLM Zoomcamp now. Is it too late for me to start?', 'Is there a deadline for joining this course?.', 'What happens if I join late but miss the project deadline?', "Does the certificate expire if I don't submit the project immediately?"]


In [31]:
# result with another prompt
result = response.output_parsed

print(result)
print(result.questions)

questions=['Can I join the LLM Zoomcamp late?', 'Do I still get a certificate if I join late?']
['Can I join the LLM Zoomcamp late?', 'Do I still get a certificate if I join late?']


In [ ]:
result = response.output_parsed

print(result)

questions=['I just discovered the course. Can I still join?']


In [43]:
from evaluation_utils import llm_structured

In [44]:
result, usage = llm_structured(
    ollama_client,
    data_gen_instructions,
    user_prompt,
    Questions
)

print(result.questions)

['I just discovered the course. Can I still join? ', 'What is the LLM Zoomcamp course?', 'How do I submit my project?', 'Is there a deadline for certificate submission?', 'Can I still enroll in the course?']


In [45]:
usage

ResponseUsage(input_tokens=286, input_tokens_details=InputTokensDetails(cached_tokens=0), output_tokens=64, output_tokens_details=OutputTokensDetails(reasoning_tokens=0), total_tokens=350)

In [46]:
from evaluation_utils import calc_price,calc_total_price

calc_price(usage)

{'input_cost': 0.0002145, 'output_cost': 0.000288, 'total_cost': 0.0005025}

In [47]:
records = []

for q in result.questions:
    records.append({
        "question": q,
        "document": doc["id"]
    })

records

[{'question': 'I just discovered the course. Can I still join? ',
  'document': '74eb249bbf'},
 {'question': 'What is the LLM Zoomcamp course?', 'document': '74eb249bbf'},
 {'question': 'How do I submit my project?', 'document': '74eb249bbf'},
 {'question': 'Is there a deadline for certificate submission?',
  'document': '74eb249bbf'},
 {'question': 'Can I still enroll in the course?', 'document': '74eb249bbf'}]

## 4.3 ground-truth-batch

In [48]:
from evaluation_utils import llm_structured_retry

def generate_ground_truth(doc):
    user_prompt = json.dumps(doc)

    out, usage = llm_structured_retry(
        ollama_client,
        data_gen_instructions,
        user_prompt,
        Questions
    )

    results = []

    for q in out.questions:
        results.append({
            "question": q,
            "document": doc["id"]
        })

    return results, usage

In [49]:
from tqdm.auto import tqdm

ground_truth = []
usages = []

for doc in tqdm(documents[:5]):
    records, usage = generate_ground_truth(doc)
    ground_truth.extend(records)
    usages.append(usage)

  0%|          | 0/5 [00:00<?, ?it/s]

In [ ]:
from concurrent.futures import ThreadPoolExecutor
from evaluation_utils import map_progress

with ThreadPoolExecutor(max_workers=6) as pool:
    results = map_progress(pool, documents, generate_ground_truth)

  0%|          | 0/79 [00:00<?, ?it/s]

In [ ]:
ground_truth = []
usages = []

for records, usage in results:
    ground_truth.extend(records)
    usages.append(usage)

len(ground_truth)

In [ ]:
from evaluation_utils import calc_price

total_cost = 0.0

for usage in usages:
    cost = calc_price(usage)
    total_cost = total_cost + cost["total_cost"]

total_cost

In [50]:
from evaluation_utils import calc_total_price

calc_total_price(usages)


0.011518500000000001

In [51]:
import pandas as pd

df_ground_truth = pd.DataFrame(ground_truth)

In [55]:
df_ground_truth.to_csv("data/ground_truth-new.csv", index=False)